# Baseline notebook

### Note: Difference between 03 and 03b
In 03, it is doing kNN based on max number of k.<br>
In 03b, it is applying kNN with cosine distance threshold and using larger k for kNN (making it loose).
In 03c, a filter to remove the three credit reporting companies from the dataset is applied.

Following is specific configuration.
- NUM_SIM:  k for kNN
- DISTANCE_THR: Distance threshold

(similarity = 1 - cosine distance)

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
SAMPLING:bool       = True
NUM_SAMPLE:int      = 250   # Number of sample data to be ingested to the graph database
SUMMARY_SAMPLE:int  = 20    # Number of samples as inputs of summarizing
RAND_SEED:int       = 77    # Seed for sampling
NUM_SIM:int         = 5     # Number of results from KNN search (does not include the own node)
EMBEDDING_MODEL:str = "text-embedding-3-small"
MAX_TOKENS:int      = 7800  # Max 8192 - some safety buffer about 5%
INDEX_NAME:str      = "idx:complaints_vss"
FILE_PATH:str       = "../data/original/complaints-2025-11-02_04_18.csv"
DB_NAME:str         = "project03"
COLLECTION_NAME:str = "complaints"

In [2]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken

In [4]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
from openai import OpenAI
import neo4j
import pymongo

In [6]:
import redis
from redis.commands.search.field import (
    NumericField,
    TagField,
    TextField,
    VectorField,
)

In [7]:
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.query import Query

In [8]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [9]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [10]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [11]:
from data_cleaning import clean_data

In [12]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_21:53:54


### Load project data and clean it up
- Clean complaints data (clean_data function)

In [13]:
df = clean_data(FILE_PATH)

In [14]:
# Remove three credit reporting companies from the dataset
df = df[
    (df["company"] != "EQUIFAX, INC.") & 
    (df["company"] != "Experian Information Solutions Inc.") &
    (df["company"] != "TRANSUNION INTERMEDIATE HOLDINGS, INC.")
    ]

In [ ]:
# Sample selection for demonstration purposes
if SAMPLING:
    df = df.sample(n=NUM_SAMPLE, random_state=RAND_SEED)    # random_state ensures reproducibility

In [16]:
df.shape

(250, 13)

In [17]:
df.head()

,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,date_sent_to_company,company_response_to_consumer,complaint_id
13366,06/08/25,Credit card,Store credit card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,TD Bank NA DBA TD Retail Card Services has vio...,Unknown,TD BANK US HOLDING COMPANY,NV,89074,06/08/25,Closed with explanation,13963767
23374,06/01/25,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,I have reason to believe Chase is violating th...,Unknown,JPMORGAN CHASE & CO.,MI,49503,06/01/25,Closed with explanation,13826021
3800,06/06/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,Subject : Formal Complaint Unauthorized Credit...,Unknown,Homespire Mortgage Corp,TX,76903,06/06/25,Closed with explanation,13938542
22433,06/02/25,Credit card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Didn't receive advertised or promotional terms,Misleading Terms Regarding Annual Fee and Welc...,Unknown,AMERICAN EXPRESS COMPANY,TX,XXXXX,06/02/25,Closed with monetary relief,13852364
11491,06/02/25,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,I previously submitted a dispute to regarding ...,Unknown,MoneyLion Inc.,FL,33054,06/02/25,Closed with non-monetary relief,13846626


In [18]:
complaints = df.to_json(orient='records')

In [19]:
complaints = json.loads(complaints)

### OpenAI

In [20]:
llm_client = OpenAI()

In [21]:
# Vectorize with Batch
def embed_batch(client, text_list, model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=text_list
    )
    return [item.embedding for item in response.data]

### Redis

In [22]:
# Note: To index the DB for vector search, db needs to be 0
redis_client = redis.Redis(host='redis', port=6379, db=0, decode_responses=True)

In [23]:
# Delete all data in the database
redis_client.flushdb()

True

In [24]:
# Store the data in Redis

In [25]:
pipeline = redis_client.pipeline()

In [26]:
for i, complaint in enumerate(complaints, start=1):
    redis_key = f"complaint:{i:03}"    # "complaint:001"
    pipeline.json().set(redis_key, "$", complaint)    # Set the whole JSON value of key redis_key to article.

In [27]:
res = pipeline.execute()

In [28]:
keys = sorted(redis_client.keys("complaint:*"))

In [29]:
text_to_be_embedded = redis_client.json().mget(keys, "$.consumer_complaint_narrative")

In [30]:
text_to_be_embedded_count = [item for sublist in text_to_be_embedded for item in sublist]

In [31]:
# Encoding used for truncating the text using tiktoken
encoding = tiktoken.encoding_for_model(EMBEDDING_MODEL)

In [32]:
# Truncate the text using tiktoken
def truncate_by_tokens(text, max_tokens=MAX_TOKENS, tokenizer=encoding):
    """
    Truncate text to a maximum number of tokens using tiktoken
    """
    tokens = tokenizer.encode(str(text))
    
    if len(tokens) <= max_tokens:
        return text    
    tokens = tokens[:max_tokens]  # Just cut off (quick and dirty solution)
    return tokenizer.decode(tokens).strip()  # Remove trailing space introduced during token cconcatenation

In [33]:
# Apply truncation to the nexted list of text
text_to_be_embedded = [
    truncate_by_tokens(item) 
    for sublist in text_to_be_embedded 
    for item in sublist
]

In [34]:
batch_size = 200
embeddings = []

for i in range(0, len(text_to_be_embedded), batch_size):
    batch = text_to_be_embedded[i:i+batch_size]
    if not batch:
        print("Skipping empty batch at i=", i)
        continue
    batch_embeddings = embed_batch(llm_client, batch, EMBEDDING_MODEL)
    embeddings.extend(batch_embeddings)

In [35]:
# Insert the vectorized descriptions to the documents in Redis using JSON.SET command
pipeline = redis_client.pipeline()
for key, embedding in zip(keys, embeddings):
    pipeline.json().set(key, "$.embeddings", embedding)
_ = pipeline.execute()  # Do not pring out all the results of from the pipeline

### Create an Index

In [36]:
VECTOR_DIMENSION = len(embeddings[0])

In [37]:
# Note: To do the embedding, we don't necessarily need to add all fields in Redis.
# For now, we are adding these in Redis for simplicity purpose, but
# in future update, we will consider adding those fields when we store data in Mongo,
# and just keep main text (consumer_complaint_narrative) and embeddings in here.
schema = (
    NumericField("$.complaint_id", as_name="complaint_id"),
    TextField("$.date_received", no_stem=True, as_name="date_received"),
    TextField("$.product", no_stem=True, as_name="product"),
    TextField("$.sub_product", as_name="sub_product"),
    TextField("$.issue", as_name="issue"),
    TextField("$.consumer_complaint_narrative", as_name="consumer_complaint_narrative"),
    TextField("$.company_public_response", as_name="company_public_response"),
    TextField("$.company", as_name="company"),
    TextField("$.state", as_name="state"),
    TextField("$.zip_code", as_name="zip_code"),
    TextField("$.date_sent_to_company", as_name="date_sent_to_company"),
    TextField("$.company_response_to_consumer", as_name="company_response_to_consumer"),
    VectorField(
        "$.embeddings",
        "FLAT",
        {
            "TYPE": "FLOAT32",
            "DIM": VECTOR_DIMENSION,
            "DISTANCE_METRIC": "COSINE",
        },
        as_name="vector",
    ),
)
definition = IndexDefinition(prefix=["complaint:"], index_type=IndexType.JSON)
res = redis_client.ft(INDEX_NAME).create_index(fields=schema, definition=definition)

### Mongo

In [38]:
# Connect to the mongo database
# mongodb is the protocol; mongo is the hostname, which for us is the container name; 27017 is the TCP port number
mongo = pymongo.MongoClient("mongodb://mongo:27017/")

In [39]:
# Drop the database
mongo.drop_database(DB_NAME)

In [40]:
# Create a new database
db = mongo[DB_NAME]

In [41]:
# Create a new collection
collection = db[COLLECTION_NAME]

### Perform vector searches in Redis and store the results to mongo

In [ ]:
# Number of results from KNN search
top_k = NUM_SIM + 1

# 1. Query to run KNN search for the item
query = (
    Query(f"(*)=>[KNN {top_k} @vector $query_vector AS vector_score]")
    .sort_by('vector_score')
    .return_fields( 
        "complaint_id",
        "title",
        "text",
        "vector_score"
    )
    .paging(0, top_k)  # Return only the top 6 hits from RediSearch
    .dialect(2)
)

for key in redis_client.scan_iter("complaint:*"):
    
    # 2. Get the embedding of one item
    embedding = redis_client.json().get(key, "$.embeddings")[0]  # Remove the doubled list
    embedding = np.array(embedding, dtype=np.float32)

    # 3. Run the query
    results = redis_client.ft(INDEX_NAME).search(
        query, {"query_vector": embedding.tobytes()}
    )

    # 4. Build a similar_items list (excluding itself)
    similar_items = []
    key_str = key.decode() if isinstance(key, bytes) else key
    
    for doc in results.docs:
        if doc.id == key_str:     # Redis key; this is the self-match
            continue
        complaint_id = getattr(doc, "complaint_id", None)
        if isinstance(complaint_id, str) and complaint_id.isdigit():
            complaint_id = int(complaint_id)
        similar_items.append({
            "complaint_id":    complaint_id,
            "similarity_score": round(1 - float(doc.vector_score), 4)  # Convert cosine distance to cosine similarity
        })
        if len(similar_items) == NUM_SIM:
            break

    # 5. Store results back into Redis JSON
    redis_client.json().set(key, "$.similar_items", similar_items)

    # 6. Get the entire record from Redis
    record = redis_client.json().get(key)

    # 7. Store the record to Mongo
    collection.insert_one(record)

### Neo4j

In [43]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [44]:
session = driver.session(database="neo4j")

In [45]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [46]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [47]:
# Erase all data in the Neo4j
my_neo4j_wipe_out_database()

In [48]:
# Create constraints
query = """
CREATE CONSTRAINT complaint_id_unique IF NOT EXISTS
FOR (c:Complaint)
REQUIRE c.complaint_id IS UNIQUE;
"""

session.run(query)

In [49]:
# Create nodes
for document in collection.find():
    complaint_id = document.get("complaint_id")
    date_received= document.get("date_received")
    product = document.get("product")
    sub_product = document.get("sub_product")
    issue = document.get("issue")
    sub_issue = document.get("sub_issue")
    consumer_complaint_narrative = document.get("consumer_complaint_narrative")
    company_public_response = document.get("company_public_response")
    company = document.get("company")
    state = document.get("state")
    zip_code = document.get("zip_code")
    date_sent_to_company = document.get("date_sent_to_company")
    company_response_to_consumer = document.get("company_response_to_consumer")
    consumer_complaint_embeddings = document.get("embeddings")

    query = """
    MERGE (c: Complaint {complaint_id: $complaint_id})
    SET 
        c.date_received = $date_received,
        c.product = $product,
        c.sub_product = $sub_product,
        c.issue = $issue,
        c.sub_issue = $sub_issue,
        c.consumer_complaint_narrative = $consumer_complaint_narrative,
        c.company_public_response = $company_public_response,
        c.company = $company,
        c.state = $state,
        c.zip_code = $zip_code,
        c.date_sent_to_company = $date_sent_to_company,
        c.company_response_to_consumer = $company_response_to_consumer,
        c.consumer_complaint_embeddings = $consumer_complaint_embeddings

    """

    session.run(query, {
        "complaint_id": complaint_id,
        "date_received": date_received,
        "product": product,
        "sub_product": sub_product,
        "issue": issue,
        "sub_issue": sub_issue,
        "consumer_complaint_narrative": consumer_complaint_narrative,
        "company_public_response": company_public_response,
        "company": company,
        "state": state,
        "zip_code": zip_code,
        "date_sent_to_company": date_sent_to_company,
        "company_response_to_consumer": company_response_to_consumer,
        "consumer_complaint_embeddings": consumer_complaint_embeddings,
    })

In [50]:
# Create relationships
for document in collection.find():
    complaint_id = document.get("complaint_id")
    similar_items = document.get("similar_items")

    for item in similar_items:
        sim_id = item.get("complaint_id")
        sim_score = item.get("similarity_score")  # This has id, title, text, and similarity_score
        sim_score = round(sim_score, 4)

        if not sim_id:
            continue

        query = """
        MATCH (com: Complaint {complaint_id: $complaint_id})
        MATCH (sim: Complaint {complaint_id: $sim_id})
        MERGE (com)-[r:SIMILAR]-(sim)
        SET r.similarity_score = $sim_score
        """

        session.run(query, {"complaint_id": complaint_id, "sim_id": sim_id, "sim_score": sim_score})

In [51]:
# Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_21:54:10
Program elapsed time: 0 minutes and 15.71 seconds
